# CNNFin Data Fetching

Run this notebook first on Jarvis Labs. It downloads and verifies all Binance 5-minute candle data required for the CNNFin experiment.

In [1]:
from pathlib import Path
import sys
import json

import pandas as pd


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "configs" / "cnnfin_5m.yaml").exists():
            return candidate
    raise FileNotFoundError("Could not find repo root containing configs/cnnfin_5m.yaml")


ROOT = find_repo_root(Path.cwd().resolve())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from cnnfin.config import load_config
from cnnfin.data import download_raw_candles, interval_to_pandas_freq, validate_cadence
from cnnfin.utils import ensure_dir, write_json

CONFIG_PATH = ROOT / "configs" / "cnnfin_5m.yaml"
base_config = load_config(CONFIG_PATH)
artifact_dir = Path(base_config.artifact_dir)
if not artifact_dir.is_absolute():
    artifact_dir = ROOT / artifact_dir
config = load_config(CONFIG_PATH, artifact_dir=str(artifact_dir))

ARTIFACT_DIR = Path(config.artifact_dir)
RAW_DIR = ARTIFACT_DIR / "raw_candles"
ZIP_DIR = ARTIFACT_DIR / "raw_zips"
REPORT_DIR = ARTIFACT_DIR / "reports"

print(f"Repo root: {ROOT}")
print(f"Config: {CONFIG_PATH}")
print(f"Artifact dir: {ARTIFACT_DIR}")
print(f"Raw candle dir: {RAW_DIR}")
print(f"ZIP cache dir: {ZIP_DIR}")

Repo root: /Users/markosmarkides/Documents/CnnFin
Config: /Users/markosmarkides/Documents/CnnFin/configs/cnnfin_5m.yaml
Artifact dir: /Users/markosmarkides/Documents/CnnFin/artifacts/cnnfin_5m
Raw candle dir: /Users/markosmarkides/Documents/CnnFin/artifacts/cnnfin_5m/raw_candles
ZIP cache dir: /Users/markosmarkides/Documents/CnnFin/artifacts/cnnfin_5m/raw_zips


## Controls

Keep `FORCE_DOWNLOAD = False` unless you explicitly want to redownload files that already exist.

In [2]:
FORCE_DOWNLOAD = False
USE_BULK_DOWNLOAD = True

print("Symbols:", config.all_symbols)
print("Interval:", config.interval)
print("Date range:", config.start_date, "to", config.end_date)
print("FORCE_DOWNLOAD:", FORCE_DOWNLOAD)
print("USE_BULK_DOWNLOAD:", USE_BULK_DOWNLOAD)

Symbols: ['BTCUSDT', 'ADAUSDT', 'BNBUSDT', 'ETHUSDT', 'LINKUSDT', 'LTCUSDT', 'SOLUSDT', 'TRXUSDT', 'XLMUSDT', 'XRPUSDT']
Interval: 5m
Date range: 2021-01-01T00:00:00Z to 2026-01-01T00:00:00Z
FORCE_DOWNLOAD: False
USE_BULK_DOWNLOAD: True


## Download Raw Candles

In [3]:
paths = download_raw_candles(config, force=FORCE_DOWNLOAD, use_bulk=USE_BULK_DOWNLOAD)
paths

BTCUSDT: using existing file /Users/markosmarkides/Documents/CnnFin/artifacts/cnnfin_5m/raw_candles/BTCUSDT_5m.pkl
ADAUSDT: using existing file /Users/markosmarkides/Documents/CnnFin/artifacts/cnnfin_5m/raw_candles/ADAUSDT_5m.pkl
BNBUSDT: using existing file /Users/markosmarkides/Documents/CnnFin/artifacts/cnnfin_5m/raw_candles/BNBUSDT_5m.pkl
ETHUSDT: using existing file /Users/markosmarkides/Documents/CnnFin/artifacts/cnnfin_5m/raw_candles/ETHUSDT_5m.pkl
LINKUSDT: using existing file /Users/markosmarkides/Documents/CnnFin/artifacts/cnnfin_5m/raw_candles/LINKUSDT_5m.pkl
LTCUSDT: using existing file /Users/markosmarkides/Documents/CnnFin/artifacts/cnnfin_5m/raw_candles/LTCUSDT_5m.pkl
SOLUSDT: using existing file /Users/markosmarkides/Documents/CnnFin/artifacts/cnnfin_5m/raw_candles/SOLUSDT_5m.pkl
TRXUSDT: using existing file /Users/markosmarkides/Documents/CnnFin/artifacts/cnnfin_5m/raw_candles/TRXUSDT_5m.pkl
XLMUSDT: using existing file /Users/markosmarkides/Documents/CnnFin/artifacts/

XRPUSDT: using existing file /Users/markosmarkides/Documents/CnnFin/artifacts/cnnfin_5m/raw_candles/XRPUSDT_5m.pkl

{'BTCUSDT': PosixPath('/Users/markosmarkides/Documents/CnnFin/artifacts/cnnfin_5m/raw_candles/BTCUSDT_5m.pkl'),
 'ADAUSDT': PosixPath('/Users/markosmarkides/Documents/CnnFin/artifacts/cnnfin_5m/raw_candles/ADAUSDT_5m.pkl'),
 'BNBUSDT': PosixPath('/Users/markosmarkides/Documents/CnnFin/artifacts/cnnfin_5m/raw_candles/BNBUSDT_5m.pkl'),
 'ETHUSDT': PosixPath('/Users/markosmarkides/Documents/CnnFin/artifacts/cnnfin_5m/raw_candles/ETHUSDT_5m.pkl'),
 'LINKUSDT': PosixPath('/Users/markosmarkides/Documents/CnnFin/artifacts/cnnfin_5m/raw_candles/LINKUSDT_5m.pkl'),
 'LTCUSDT': PosixPath('/Users/markosmarkides/Documents/CnnFin/artifacts/cnnfin_5m/raw_candles/LTCUSDT_5m.pkl'),
 'SOLUSDT': PosixPath('/Users/markosmarkides/Documents/CnnFin/artifacts/cnnfin_5m/raw_candles/SOLUSDT_5m.pkl'),
 'TRXUSDT': PosixPath('/Users/markosmarkides/Documents/CnnFin/artifacts/cnnfin_5m/raw_candles/TRXUSDT_5m.pkl'),
 'XLMUSDT': PosixPath('/Users/markosmarkides/Documents/CnnFin/artifacts/cnnfin_5m/raw_candles/XLMUSDT_

## Verify Downloaded Files

In [4]:
freq = interval_to_pandas_freq(config.interval)
verification_rows = []

for symbol in config.all_symbols:
    path = ARTIFACT_DIR / "raw_candles" / f"{symbol}_{config.interval}.pkl"
    if not path.exists():
        raise FileNotFoundError(f"Missing raw candle file for {symbol}: {path}")

    df = pd.read_pickle(path)
    cadence = validate_cadence(df, time_col="Open time", freq=freq)
    verification_rows.append(
        {
            "symbol": symbol,
            "path": str(path),
            "rows": int(len(df)),
            "start": str(pd.to_datetime(df["Open time"], utc=True).min()) if len(df) else None,
            "end": str(pd.to_datetime(df["Open time"], utc=True).max()) if len(df) else None,
            "duplicates": int(cadence["duplicates"]),
            "expected_rows": int(cadence.get("expected_rows", 0)),
            "missing_count": int(cadence["missing_count"]),
            "missing_examples": cadence.get("missing_examples", []),
        }
    )

verification = pd.DataFrame(verification_rows)
ensure_dir(REPORT_DIR)
verification_csv = REPORT_DIR / "raw_candle_verification.csv"
verification_json = REPORT_DIR / "raw_candle_verification.json"
verification.to_csv(verification_csv, index=False)
write_json(verification_rows, verification_json)

if verification["rows"].eq(0).any():
    raise ValueError("At least one downloaded candle file is empty.")
if verification["duplicates"].gt(0).any():
    raise ValueError("At least one downloaded candle file has duplicate timestamps.")

print(f"Saved verification CSV: {verification_csv}")
print(f"Saved verification JSON: {verification_json}")
display(verification.drop(columns=["missing_examples"]))

Saved verification CSV: /Users/markosmarkides/Documents/CnnFin/artifacts/cnnfin_5m/reports/raw_candle_verification.csv
Saved verification JSON: /Users/markosmarkides/Documents/CnnFin/artifacts/cnnfin_5m/reports/raw_candle_verification.json


,symbol,path,rows,start,end,duplicates,expected_rows,missing_count
0,BTCUSDT,/Users/markosmarkides/Documents/CnnFin/artifac...,525675,2021-01-01 00:00:00+00:00,2025-12-31 23:55:00+00:00,0,525888,213
1,ADAUSDT,/Users/markosmarkides/Documents/CnnFin/artifac...,525675,2021-01-01 00:00:00+00:00,2025-12-31 23:55:00+00:00,0,525888,213
2,BNBUSDT,/Users/markosmarkides/Documents/CnnFin/artifac...,525675,2021-01-01 00:00:00+00:00,2025-12-31 23:55:00+00:00,0,525888,213
3,ETHUSDT,/Users/markosmarkides/Documents/CnnFin/artifac...,525675,2021-01-01 00:00:00+00:00,2025-12-31 23:55:00+00:00,0,525888,213
4,LINKUSDT,/Users/markosmarkides/Documents/CnnFin/artifac...,525675,2021-01-01 00:00:00+00:00,2025-12-31 23:55:00+00:00,0,525888,213
5,LTCUSDT,/Users/markosmarkides/Documents/CnnFin/artifac...,525675,2021-01-01 00:00:00+00:00,2025-12-31 23:55:00+00:00,0,525888,213
6,SOLUSDT,/Users/markosmarkides/Documents/CnnFin/artifac...,525675,2021-01-01 00:00:00+00:00,2025-12-31 23:55:00+00:00,0,525888,213
7,TRXUSDT,/Users/markosmarkides/Documents/CnnFin/artifac...,525675,2021-01-01 00:00:00+00:00,2025-12-31 23:55:00+00:00,0,525888,213
8,XLMUSDT,/Users/markosmarkides/Documents/CnnFin/artifac...,525675,2021-01-01 00:00:00+00:00,2025-12-31 23:55:00+00:00,0,525888,213
9,XRPUSDT,/Users/markosmarkides/Documents/CnnFin/artifac...,525675,2021-01-01 00:00:00+00:00,2025-12-31 23:55:00+00:00,0,525888,213


## Ready For Dataset/Image Build

After this notebook finishes, run `exploration/image_builder.ipynb`.